In [0]:
select * from tpch_sales.gold.vw_annual_sales_kpi

In [0]:
select
  year(order_date)  as sales_year,
  month(order_date) as sales_month,
  round(sum(extended_price * (1 - discount)), 2) as monthly_net_revenue
from tpch_sales.gold.fact_sales
group by 1, 2
order by 1, 2

In [0]:
select
  sales_year,
  round(net_revenue, 2) as net_revenue,
  round(net_revenue - lag(net_revenue) over (order by sales_year), 2) as yoy_change,
  round(100 * (net_revenue - lag(net_revenue) over (order by sales_year))
        / lag(net_revenue) over (order by sales_year), 2) as yoy_change_pct
from tpch_sales.gold.vw_annual_sales_kpi
order by sales_year

In [0]:
select
  c.market_segment,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue,
  count(distinct f.order_key) as orders
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_customer c
  on f.customer_key = c.customer_key
group by 1
order by net_revenue desc

In [0]:
select
  f.customer_key,
  c.customer_name,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_customer c
  on f.customer_key = c.customer_key
group by 1, 2
order by net_revenue desc
limit 10

In [0]:
with customer_revenue as (
  select
    f.customer_key,
    c.customer_name,
    sum(f.extended_price * (1 - f.discount)) as net_revenue
  from tpch_sales.gold.fact_sales f
  join tpch_sales.gold.dim_customer c
    on f.customer_key = c.customer_key
  group by 1, 2
),
ranked as (
  select *, rank() over (order by net_revenue desc) as rnk
  from customer_revenue
),
totals as (
  select sum(net_revenue) as total_revenue from customer_revenue
)
select
  'TOP 10 COMBINED' as customer_name,
  null as customer_key,
  round(sum(r.net_revenue), 2) as net_revenue,
  round(100 * sum(r.net_revenue) / max(t.total_revenue), 4) as share_pct
from ranked r cross join totals t
where r.rnk <= 10
union all
select
  c.customer_name,
  c.customer_key,
  round(c.net_revenue, 2) as net_revenue,
  round(100 * c.net_revenue / t.total_revenue, 4) as share_pct
from ranked c cross join totals t
where c.rnk <= 10
order by net_revenue desc

In [0]:
select
  p.product_key,
  p.product_name,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_product p
  on f.product_key = p.product_key
group by 1, 2
order by net_revenue desc
limit 10

In [0]:
select
  p.product_key,
  p.product_name,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_product p
  on f.product_key = p.product_key
group by 1, 2
order by net_revenue asc
limit 10

In [0]:
select
  p.product_type,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue,
  count(distinct p.product_key) as products
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_product p
  on f.product_key = p.product_key
group by 1
order by net_revenue desc

In [0]:
select
  s.supplier_key,
  s.supplier_name,
  round(sum(f.extended_price * (1 - f.discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_supplier s
  on f.supplier_key = s.supplier_key
group by 1, 2
order by net_revenue desc
limit 10

In [0]:
with supplier_revenue as (
  select
    f.supplier_key,
    s.supplier_name,
    sum(f.extended_price * (1 - f.discount)) as net_revenue
  from tpch_sales.gold.fact_sales f
  join tpch_sales.gold.dim_supplier s
    on f.supplier_key = s.supplier_key
  group by 1, 2
),
ranked as (
  select *, rank() over (order by net_revenue desc) as rnk
  from supplier_revenue
),
totals as (
  select sum(net_revenue) as total_revenue from supplier_revenue
)
select
  'TOP 10 COMBINED' as supplier_name,
  null as supplier_key,
  round(sum(r.net_revenue), 2) as net_revenue,
  round(100 * sum(r.net_revenue) / max(t.total_revenue), 4) as share_pct
from ranked r cross join totals t
where r.rnk <= 10
union all
select
  s.supplier_name,
  s.supplier_key,
  round(s.net_revenue, 2) as net_revenue,
  round(100 * s.net_revenue / t.total_revenue, 4) as share_pct
from ranked s cross join totals t
where s.rnk <= 10
order by net_revenue desc

In [0]:
select
  customer_region_name as region,
  round(sum(extended_price * (1 - discount)), 2) as net_revenue,
  count(distinct order_key) as orders
from tpch_sales.gold.fact_sales
group by 1
order by net_revenue desc

In [0]:
select
  customer_nation_name as nation,
  customer_region_name as region,
  round(sum(extended_price * (1 - discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales
group by 1, 2
order by net_revenue desc

In [0]:
select
  return_flag,
  round(sum(extended_price * (1 - discount)), 2) as net_revenue,
  sum(quantity) as total_quantity,
  count(*) as line_items,
  round(100 * sum(extended_price * (1 - discount))
        / sum(sum(extended_price * (1 - discount))) over (), 2) as revenue_share_pct
from tpch_sales.gold.fact_sales
group by 1
order by 1

In [0]:
select
  c.market_segment,
  round(avg(f.discount), 4) as avg_discount
from tpch_sales.gold.fact_sales f
join tpch_sales.gold.dim_customer c
  on f.customer_key = c.customer_key
group by 1
order by avg_discount desc

In [0]:
select
  ship_mode,
  round(avg(datediff(receipt_date, ship_date)), 2) as avg_ship_to_receipt_days
from tpch_sales.gold.fact_sales
group by 1
order by avg_ship_to_receipt_days desc

In [0]:
select
  order_priority,
  count(distinct order_key) as orders,
  round(sum(extended_price * (1 - discount)), 2) as net_revenue
from tpch_sales.gold.fact_sales
group by 1
order by orders desc

In [0]:
select
  year(order_date)  as sales_year,
  month(order_date) as sales_month,
  order_status,
  count(distinct order_key) as orders
from tpch_sales.gold.fact_sales
group by 1, 2, 3
order by 1, 2, 3

In [0]:
with product_segment_revenue as (
  select
    c.market_segment,
    p.product_key,
    p.product_name,
    sum(f.extended_price * (1 - f.discount)) as net_revenue
  from tpch_sales.gold.fact_sales f
  join tpch_sales.gold.dim_customer c
    on f.customer_key = c.customer_key
  join tpch_sales.gold.dim_product p
    on f.product_key = p.product_key
  group by 1, 2, 3
)
select
  market_segment,
  product_key,
  product_name,
  round(net_revenue, 2) as net_revenue,
  rank() over (partition by market_segment order by net_revenue desc) as rank_in_segment
from product_segment_revenue
order by market_segment, rank_in_segment

In [0]:
with customer_geo as (
  select
    customer_region_name as region,
    sum(extended_price * (1 - discount)) as customer_geography_revenue
  from tpch_sales.gold.fact_sales
  group by 1
),
supplier_geo as (
  select
    supplier_region_name as region,
    sum(extended_price * (1 - discount)) as supplier_geography_revenue
  from tpch_sales.gold.fact_sales
  group by 1
)
select
  coalesce(c.region, s.region) as region,
  round(c.customer_geography_revenue, 2) as revenue_by_customer_region,
  round(s.supplier_geography_revenue, 2) as revenue_by_supplier_region,
  round(c.customer_geography_revenue - s.supplier_geography_revenue, 2) as difference
from customer_geo c
full outer join supplier_geo s
  on c.region = s.region
order by region